# Esplorazione file Excel — Cesvi Indice Infanzia\n\nQuesto notebook carica tutti e 6 i file Excel presenti in `data/original/tables/`\ne ne analizza la struttura: fogli, colonne, prime righe, tipi di dato.\n\nObiettivo: capire la struttura dati prima di scrivere `build_dataset.py`.

In [ ]:
import pandas as pd
from pathlib import Path

TABLES_DIR = Path("original/tables")
excel_files = sorted(TABLES_DIR.glob("*.xlsx"))
print(f"File trovati: {len(excel_files)}")
for f in excel_files:
    print(" -", f.name)

## 1. Fogli disponibili in ogni file

In [ ]:
sheets_map = {}
for f in excel_files:
    xl = pd.ExcelFile(f)
    sheets_map[f.name] = xl.sheet_names
    print(f"\n{'='*60}")
    print(f"FILE: {f.name}")
    print(f"  Fogli ({len(xl.sheet_names)}): {xl.sheet_names}")

## 2. Struttura di ogni foglio (colonne, shape, prime 5 righe)

In [ ]:
all_sheets_data = {}  # { filename: { sheet_name: df } }

for f in excel_files:
    all_sheets_data[f.name] = {}
    print(f"\n{'#'*70}")
    print(f"FILE: {f.name}")
    for sheet in sheets_map[f.name]:
        df = pd.read_excel(f, sheet_name=sheet, header=0)
        all_sheets_data[f.name][sheet] = df
        print(f"\n  --- Foglio: '{sheet}' | Shape: {df.shape} ---")
        print(f"  Colonne: {list(df.columns)}")
        print(f"  Tipi:\n{df.dtypes.to_string()}")
        print(f"\n  Prime 5 righe:")
        display(df.head())

## 3. Riepilogo colonne comuni tra anni

Identifica i nomi colonna presenti in TUTTI i file (intersezione) e quelli presenti solo in alcuni (unione).

In [ ]:
# Usa il primo foglio di ogni file come foglio principale
def first_sheet_df(fname):
    sheets = sheets_map[fname]
    return all_sheets_data[fname][sheets[0]]

col_sets = {fname: set(first_sheet_df(fname).columns) for fname in all_sheets_data}

all_cols = set.union(*col_sets.values())
common_cols = set.intersection(*col_sets.values())
only_in_some = all_cols - common_cols

print(f"Colonne comuni a TUTTI i file ({len(common_cols)}):")
for c in sorted(common_cols):
    print(f"  - {c}")

print(f"\nColonne presenti solo in alcuni file ({len(only_in_some)}):")
for c in sorted(only_in_some):
    present_in = [fname for fname, cols in col_sets.items() if c in cols]
    print(f"  - '{c}'  →  {present_in}")

## 4. Valori unici per la colonna territorio (prima colonna testuale)

In [ ]:
for fname in all_sheets_data:
    df = first_sheet_df(fname)
    # Cerca la prima colonna object/string come possibile colonna territorio
    text_cols = [c for c in df.columns if df[c].dtype == object]
    print(f"\n{fname}")
    if text_cols:
        col = text_cols[0]
        print(f"  Prima colonna testuale: '{col}'")
        print(f"  Valori unici ({df[col].nunique()}): {sorted(df[col].dropna().unique().tolist())}")
    else:
        print("  Nessuna colonna testuale trovata")

## 5. Statistiche descrittive delle colonne numeriche (min, max, media, std)

In [ ]:
for fname in all_sheets_data:
    df = first_sheet_df(fname)
    num_df = df.select_dtypes(include="number")
    print(f"\n{fname} — colonne numeriche: {list(num_df.columns)}")
    display(num_df.describe().round(3))